Here i am going to try to use the backtest .py tool box rahter than bt 

In [ ]:
import yfinance as yf
import pandas as pd
from backtesting import Backtest, Strategy
import math

# Define the tickers and the date range
tickers = 'GLD'
start_date = '2018-01-01'
end_date = '2023-09-01'

# Download the full OHLCV data
data = yf.download(tickers, start=start_date, end=end_date)

# Ensure the DataFrame has the required columns
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

# Convert the index to datetime if it's not already
data.index = pd.to_datetime(data.index)


class DCA(Strategy):

    amount_to_invest = 500

    def init(self):
        self.day_of_week = self.I(lambda: self.data.index.dayofweek)

    def next(self):
        if self.day_of_week[-1] == 1:  # Buy on Tuesday (0=Monday, 1=Tuesday, ...)
            size = math.floor(self.amount_to_invest / self.data.Close[-1])
            if size > 0:
                self.buy(size=size)


bt = Backtest(
    data,
    DCA,
    trade_on_close=True,
    commission=0.02,
)

stats = bt.run()

print(stats)

# Size  EntryBar  ExitBar  EntryPrice  ExitPrice 
trades = stats["_trades"]
price_paid = trades["Size"] * trades["EntryPrice"]
total_invested = price_paid.sum()

current_shares = trades["Size"].sum()
current_equity = current_shares * data['Close'].iloc[-1]

print("Total investment:", total_invested)
print("Current Shares:", current_shares)
print("Current Equity:", current_equity)

print("Return:", current_equity / total_invested)

[*********************100%***********************]  2 of 2 completed


Price                            Open                   High             \
Ticker                            GLD        IOO         GLD        IOO   
Date                                                                      
2018-01-02 00:00:00+00:00  124.660004  46.509998  125.180000  46.685001   
2018-01-03 00:00:00+00:00  125.050003  46.695000  125.089996  46.930000   
2018-01-04 00:00:00+00:00  124.889999  47.110001  125.849998  47.330002   
2018-01-05 00:00:00+00:00  124.930000  47.455002  125.480003  47.544998   
2018-01-08 00:00:00+00:00  125.199997  47.540001  125.320000  47.595001   

Price                             Low                  Close             \
Ticker                            GLD        IOO         GLD        IOO   
Date                                                                      
2018-01-02 00:00:00+00:00  124.389999  46.490002  125.150002  46.685001   
2018-01-03 00:00:00+00:00  124.099998  46.650002  124.820000  46.849998   
2018-01-04 00:00:00+00:0

TypeError: other must be a MultiIndex or a list of tuples